In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
team_name="team_lemma"
catalog_name=f"charles_schwab_retailbrokerage_dev_{team_name}"
dbutils.widgets.text("batch_id","1","BATCH ID")
bronze_watchhistory=f"{catalog_name}.bronze.watchhistory"
silver_watches=f"{catalog_name}.silver.watches"

In [0]:
batch_id=dbutils.widgets.get("batch_id")

In [0]:
df_bronze=spark.read.table(bronze_watchhistory)

In [0]:
# df_bronze.printSchema()

In [0]:
#type casting
df_bronze=df_bronze.withColumn("W_DTS",col("W_DTS").cast( "timestamp"))
df_bronze=df_bronze.withColumn("W_C_ID",col("W_C_ID").cast("bigint"))

In [0]:
#Batch 1 Not contain the CDC Flag
if "CDC_FLAG" in df_bronze.columns:
    df_bronze = df_bronze.withColumn("CDC_FLAG", coalesce(col("CDC_FLAG"), lit("I")))
else:
    df_bronze = df_bronze.withColumn("CDC_FLAG", lit("I"))

In [0]:
#Deduplication
df_bronze.createOrReplaceTempView("df_bronze")
df_dedup=spark.sql(f"""
          SELECT * EXCEPT (rn) 
                        from (
                            SELECT * ,row_number() over(
                                PARTITION BY W_C_ID,W_S_SYMB,W_DTS
                                ORDER BY _ingest_ts DESC
                            ) As rn
                            FROM df_bronze
                        ) where rn=1
          """)
# df_dedup.count()

#droping ingest ts
df_dedup=df_dedup.drop("_ingest_ts")

#adding load ts
df_dedup=df_dedup.withColumn("_load_ts",current_timestamp())
df_dedup.createOrReplaceTempView("df_dedup")

In [0]:
# watchhistory is full rebuild so it will overwrite the silver table
try:
    print("Rebuild process start....")
    df_dedup.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_watches)
    print("Rebuild process completed....")

    run_id=df_dedup.select("_run_id").first()[0]
    source_count = spark.read.table(bronze_watchhistory).count()
    silver_history = spark.sql(f"DESCRIBE HISTORY {silver_watches}").first()
    target_count = int(silver_history["operationMetrics"].get("numOutputRows", 0))

    log_pipeline_recon(
        spark=spark,
        run_id=run_id,
        batch_id=batch_id,
        domain="CUSTOMER",
        table_name="watches",    
        source_layer="bronze",    
        target_layer="silver",
        source_count=source_count,
        target_count=target_count
    )
        
       
    log_audit_event(
        spark=spark,
        run_id=run_id,
        batch=batch_id,
        layer="silver",
        table_name="watches",
        operation="OVERWRITE",      
        rows_affected=target_count
    )
except Exception as e:
    print("Failed to rebuild")
    raise e


In [0]:
# spark.table(silver_watches).count()

In [0]:
# spark.table(silver_watches).limit(10).display()